<a href="https://colab.research.google.com/github/Aggarwalmansi/GENAI/blob/main/Essay_writing_AIAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# 1. Install dependencies
!pip install -q U langgraph langchain langchain-groq

import os
from google.colab import userdata
from typing import TypedDict, List
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# 2. Setup Groq LLM using Colab Secrets
# Make sure you've added "GROQ_API_KEY" in the 'Keys' (🔑) tab in Colab
os.environ["GROQ_API_KEY"] = userdata.get('groq_api')

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

In [10]:
# --- 1. Define the Graph State ---
class AgentState(TypedDict):
    topic: str
    key_terms: List[str]
    research_data: str
    draft: str
    review_feedback: str
    final_essay: str

# --- 2. Step 1 Node: Generate Key Terms ---
def generate_key_terms(state: AgentState):
    print("--- GENERATING KEY TERMS ---")
    topic = state['topic']

    prompt = f"""
    You are an expert Research Librarian. Your goal is to break down the topic: "{topic}"
    into a comprehensive search strategy for a high-scoring academic essay.

    STEP 1: Identify the following dimensions of the topic:
    - Historical Context & Origins
    - Key Current Statistics/Data points
    - Major Controversies or Differing Perspectives
    - Future Implications

    STEP 2: Based on those dimensions, generate 6-8 precise, high-intent search queries.
    Focus on finding "Evidence," "Case Studies," and "Expert Opinions."

    OUTPUT FORMAT:
    Return ONLY a comma-separated list of the search queries.
    Example: [term 1, term 2, term 3]
    """

    response = llm.invoke(prompt)
    # Clean and split the response into a list
    terms = [term.strip() for term in response.content.split(",")]

    return {"key_terms": terms}

# --- 3. Initialize the Graph ---
workflow = StateGraph(AgentState)

# Add our first node
workflow.add_node("generate_terms", generate_key_terms)

# Set the entry point
workflow.set_entry_point("generate_terms")
workflow.add_edge("generate_terms", END) # For now, we end here to test Step 1

# Compile the app
app = workflow.compile()

In [7]:
!pip install -q duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 42.5 MB/s eta 0:00:00


In [3]:
!pip install -U duckduckgo-search

In [11]:
from langchain_community.tools import DuckDuckGoSearchResults

# Using SearchResults instead of SearchRun often bypasses the internal import bug
search = DuckDuckGoSearchResults()

def perform_research(state: AgentState):
    print("---🔍 STEP 2: PERFORMING WEB RESEARCH ---")
    queries = state['key_terms']
    aggregated_research = ""

    for query in queries:
        print(f"Searching for: {query}...")
        try:
            # We use .invoke() which is the standard for LangChain 0.2/0.3
            results = search.invoke(query)
            aggregated_research += f"\n\nSource (Query: {query}):\n{results}"
        except Exception as e:
            print(f"Search failed for {query}: {e}")
            continue

    return {"research_data": aggregated_research}

In [7]:
import sys
!{sys.executable} -m pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.7 MB/s eta 0:00:00


In [12]:
def write_draft(state: AgentState):
    print("--- ✍️ STEP 3: WRITING ELITE FIRST DRAFT ---")
    topic = state['topic']
    research = state['research_data']

    # This prompt uses "Role Prompting" and "Step-by-Step" instructions
    prompt = f"""
    SYSTEM ROLE: You are a Pulitzer Prize-winning essayist and Academic Dean. Your writing is known for its clarity, sophisticated vocabulary, and compelling logic.

    TOPIC: {topic}
    RESEARCH PROVIDED:
    {research}

    TASK: Write a definitive, high-level essay based on the research above.

    GOALS:
    1. THESIS: Formulate a sophisticated thesis statement in the introduction that goes beyond the obvious.
    2. PARAGRAPH STRUCTURE: Use the PEEL method:
       - Point: Start with a clear topic sentence.
       - Evidence: Support with specific data from the research.
       - Explanation: Analyze why this evidence matters.
       - Link: Transition smoothly to the next point.
    3. VOCABULARY: Use precise, academic language (avoid "good," "bad," "big," "thing").
    4. COUNTER-ARGUMENT: Acknowledge at least one opposing view or complexity to show depth.
    5. CONCLUSION: Do not just summarize; synthesize the arguments to leave a lasting impact.

    CONSTRAINTS:
    - Total word count: 800-1200 words.
    - No "AI filler" phrases like "In the fast-paced world of today..."
    - Start with a compelling hook.

    WRITE THE ESSAY:
    """

    response = llm.invoke(prompt)

    return {"draft": response.content}

In [16]:
# --- 1. Step 4 Node: The "Auditor" (Upgraded Review Node) ---
def review_draft(state: AgentState):
    print("--- 🧐 STEP 4: RIGOROUS ACADEMIC AUDIT ---")
    draft = state['draft']
    topic = state['topic']

    prompt = f"""
    ROLE: You are the Lead Auditor for a top-tier academic journal (Nature/Harvard Business Review).
    Your goal is to tear this draft apart to find any weaknesses.

    TOPIC: "{topic}"
    DRAFT TO AUDIT:
    {draft}

    AUDIT PROTOCOL:
    1. LOGIC CHECK: Identify any "unearned" conclusions or logical leaps.
    2. REDUNDANCY CHECK: Find repetitive AI-phrases (e.g., "In conclusion," "it's important to remember," "delve into").
    3. SOPHISTICATION INDEX: Is the vocabulary too simple? Is the sentence structure too repetitive?
    4. DATA INTEGRATION: Did the writer actually use the research, or did they just summarize general knowledge?
    5. THE "SO WHAT?" FACTOR: Does the conclusion offer a unique insight or just repeat the intro?

    OUTPUT REQUIREMENTS:
    - Provide a "Delete List" (specific phrases/sentences to remove).
    - Provide a "Deepen List" (where more specific data or analysis is needed).
    - Rate the draft from 1-10. If it is below 9, explain exactly why.
    """

    response = llm.invoke(prompt)
    return {"review_feedback": response.content}

# --- 2. Step 5 Node: The "Master Synthesizer" (Upgraded Final Node) ---
def finalize_essay(state: AgentState):
    print("--- 🖋️ STEP 5: MASTER SYNTHESIS & FINAL POLISH ---")
    draft = state['draft']
    feedback = state['review_feedback']
    topic = state['topic']

    prompt = f"""
    ROLE: You are a Master Synthesizer. You do not just "edit"; you transform.
    You have a draft and a brutal academic audit.

    MISSION:
    Rewrite the draft into a world-class essay that would pass the most rigorous editorial review.

    INPUTS:
    - DRAFT: {draft}
    - AUDIT FEEDBACK: {feedback}

    STRICT GUIDELINES:
    1. ELIMINATE CLICHÉS: If a sentence sounds like a typical AI response, rewrite it to be sharper and more human.
    2. VARY SENTENCE LENGTH: Mix short, punchy sentences with complex, rhythmic ones.
    3. ELEVATE TRANSITIONS: Use sophisticated transitions (e.g., "Conversely," "Implicit in this argument is," "This paradigm shifts when...").
    4. NO FILLER: Every single word must earn its place on the page.

    Produce the final, definitive masterpiece on "{topic}":
    """

    response = llm.invoke(prompt)
    return {"final_essay": response.content}

# --- 3. RE-INITIALIZE THE GRAPH ---
builder = StateGraph(AgentState)

builder.add_node("generate_terms", generate_key_terms)
builder.add_node("research_topic", perform_research)
builder.add_node("write_draft", write_draft)
builder.add_node("review_draft", review_draft)
builder.add_node("finalize_essay", finalize_essay)

builder.set_entry_point("generate_terms")
builder.add_edge("generate_terms", "research_topic")
builder.add_edge("research_topic", "write_draft")
builder.add_edge("write_draft", "review_draft")
builder.add_edge("review_draft", "finalize_essay")
builder.add_edge("finalize_essay", END)

essay_ai = builder.compile()
print("✅ Elite Essay Graph Compiled!")

✅ Elite Essay Graph Compiled!


In [17]:
inputs = {"topic": "The ethical implications of neural implants in 2030"}

for output in essay_ai.stream(inputs):
    for key, value in output.items():
        print(f"\n--- Finished Node: {key} ---")
        if key == "finalize_essay":
            print("\n🌟 FINAL RESULT 🌟\n")
            print(value["final_essay"])

--- GENERATING KEY TERMS ---

--- Finished Node: generate_terms ---
---🔍 STEP 2: PERFORMING WEB RESEARCH ---
Searching for: neural implants history and development...
Searching for: current statistics on neural implant adoption rates...
Searching for: ethical concerns surrounding neural implants and human enhancement...
Searching for: case studies on neural implant safety and efficacy...
Searching for: expert opinions on neural implant regulation and governance...
Searching for: future implications of neural implants on human identity and society...
Searching for: neural implant technology and its potential impact on social inequality...
Searching for: evidence of neural implant benefits and risks in clinical trials...
Searching for: neural implant industry trends and market analysis 2030...

--- Finished Node: research_topic ---
--- ✍️ STEP 3: WRITING ELITE FIRST DRAFT ---

--- Finished Node: write_draft ---
--- 🧐 STEP 4: RIGOROUS ACADEMIC AUDIT ---

--- Finished Node: review_draft --

In [18]:
inputs = {"topic": "The socio-economic impact of universal basic income by 2030"}

for output in essay_ai.stream(inputs):
    for key, value in output.items():
        print(f"\n✅ COMPLETED: {key}")

        if key == "review_draft":
            print("\n--- 🔥 THE AUDITOR'S VERDICT ---")
            print(value["review_feedback"])

        if key == "finalize_essay":
            print("\n--- 🏁 FINAL TRANSFORMATION ---")
            final = value["final_essay"]
            # Check if the final result is significantly different
            print(f"Final Essay Length: {len(final.split())} words.")
            print(final[:500] + "...") # Print the first 500 characters

--- GENERATING KEY TERMS ---

✅ COMPLETED: generate_terms
---🔍 STEP 2: PERFORMING WEB RESEARCH ---
Searching for: universal basic income historical context...
Searching for: universal basic income pilot programs statistics...
Searching for: universal basic income effectiveness case studies...
Searching for: expert opinions on universal basic income implementation...
Searching for: universal basic income controversy and criticism...
Searching for: universal basic income future implications by 2030...
Searching for: universal basic income economic impact analysis...
Searching for: universal basic income social impact research studies...

✅ COMPLETED: research_topic
--- ✍️ STEP 3: WRITING ELITE FIRST DRAFT ---

✅ COMPLETED: write_draft
--- 🧐 STEP 4: RIGOROUS ACADEMIC AUDIT ---

✅ COMPLETED: review_draft

--- 🔥 THE AUDITOR'S VERDICT ---
**Audit Report**

**LOGIC CHECK:**
The draft makes several unearned conclusions, such as the statement that UBI can "provide a safety net for the most vuln

In [22]:
def finalize_essay(state: AgentState):
    print("--- 🖋️ STEP 5: MASTER RE-SYNTHESIS (FINAL POLISH) ---")

    # Extracting the three pillars of the final version
    original_topic = state['topic']
    first_draft = state['draft']
    critique = state['review_feedback']

    prompt = f"""
    ROLE: You are a World-Class Editor-in-Chief at a top-tier publication (like The Economist or Atlantic Monthly).
    Your mission is to take a 'rough' first draft and the Auditor's brutal critique to produce a definitive masterpiece.

    TOPIC: {original_topic}

    --- THE RAW MATERIAL (FIRST DRAFT) ---
    {first_draft}

    --- THE AUDIT REPORT (CRITIQUE TO IMPLEMENT) ---
    {critique}

    TRANSFORMATION MANDATE:
    1. ADDRESS ALL CRITIQUES: Every weakness identified by the Auditor MUST be solved.
    2. VOICE & VIBE: Strip away 'AI-filler' (e.g., 'In conclusion,' 'It is important to note'). Replace with punchy, authoritative prose.
    3. SENTENCE ARCHITECTURE: Use 'The Hemingway/Faulkner Balance'—mix short, impactful sentences with elegant, complex ones.
    4. DATA DENSITY: Ensure the research facts are woven into the narrative, not just listed.
    5. FLOW: Create 'Invisible Bridges'—the end of one paragraph should naturally pull the reader into the next.

    FINAL INSTRUCTION:
    Do not just edit. REWRITE the essay from scratch if necessary to achieve a 10/10 quality score.
    The final result must be indistinguishable from a high-level human expert's work.

    PRODUCE THE FINAL MASTERPIECE BELOW:
    """

    response = llm.invoke(prompt)

    # Save the masterpiece to the state
    return {"final_essay": response.content}

In [23]:
# 1. Define your input topic
inputs = {"topic": "The ethical implications of neural implants in 2030"}

# 2. Run the graph and STORE it in 'result'
# .invoke() runs the whole thing and returns the final state
print("🚀 Starting the Agent... (This may take a minute)")
result = essay_ai.invoke(inputs)

# 3. Now the Quality Metrics code will work!
initial = result['draft']
final = result['final_essay']

print(f"\n--- 📊 QUALITY METRICS ---")
print(f"Draft 1 Length: {len(initial.split())} words")
print(f"Final Masterpiece Length: {len(final.split())} words")

# Check for improvement
if len(final) > 0:
    print("✅ Final essay generated successfully.")

# Display a snippet of the Final Result
print("\n--- 🌟 PREVIEW OF FINAL MASTERPIECE ---\n")
print(final[:1000] + "...")

🚀 Starting the Agent... (This may take a minute)
--- GENERATING KEY TERMS ---
---🔍 STEP 2: PERFORMING WEB RESEARCH ---
Searching for: neural implants history and development...
Searching for: current statistics on neural implant adoption rates...
Searching for: ethical concerns surrounding neural implants and privacy...
Searching for: expert opinions on neural implant regulation...
Searching for: case studies of neural implant successes and failures...
Searching for: future implications of neural implants on human cognition...
Searching for: neural implant controversy and public perception...
Searching for: "neural implants and informed consent"...
--- ✍️ STEP 3: WRITING ELITE FIRST DRAFT ---
--- 🧐 STEP 4: RIGOROUS ACADEMIC AUDIT ---
--- 🖋️ STEP 5: MASTER SYNTHESIS & FINAL POLISH ---

--- 📊 QUALITY METRICS ---
Draft 1 Length: 777 words
Final Masterpiece Length: 875 words
✅ Final essay generated successfully.

--- 🌟 PREVIEW OF FINAL MASTERPIECE ---

The advent of neural implants has pre